# 06 — Entanglement entropy and the holographic prerequisites

**Spacetime Lab — Phase 6 concept + implementation notebook**

This is the **conceptual pivot** of the roadmap. For five phases we have been doing classical general relativity. Phase 6 enters quantum information theory. Phase 7+ will weld the two together — that is the **holography** programme — and the bridge is already visible in our previous phases: the **Bekenstein-Hawking entropy** $S_{BH} = A/4$ that we computed for Schwarzschild and Kerr is *literally* an entanglement entropy in disguise.

**What you will learn**

1. Density matrices, partial traces, and why mixed states arise from tracing out a subsystem of a pure state
2. The von Neumann entropy $S(\rho) = -\text{tr}(\rho \log \rho)$, its bounds, and its physical meaning
3. The entanglement entropy of a bipartition: $S_A = S(\rho_A)$ where $\rho_A = \text{tr}_B(|\psi\rangle\langle\psi|)$
4. The Schmidt decomposition: every pure bipartite state can be written $|\psi\rangle = \sum_i \lambda_i |u_i\rangle|v_i\rangle$, computed via the SVD
5. Verification against canonical analytic values: Bell pair = $\log 2$, GHZ = $\log 2$, maximally mixed = $\log d$, product state = 0
6. Quantum mutual information $I(A:B) = S_A + S_B - S_{AB}$
7. The teaser link to holography: how Bekenstein-Hawking entropy $S = A/4$ becomes a Ryu-Takayanagi statement in Phases 7-9

**References**

- Nielsen & Chuang, *Quantum Computation and Quantum Information*, ch. 2 (Schmidt) and ch. 11 (entropy)
- Calabrese & Cardy, *Entanglement entropy and quantum field theory*, J. Stat. Mech. 0406:P06002 (2004) — the area law for QFT
- Wikipedia: [Entropy of entanglement](https://en.wikipedia.org/wiki/Entropy_of_entanglement), [Von Neumann entropy](https://en.wikipedia.org/wiki/Von_Neumann_entropy)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.entropy import (
    density_matrix,
    partial_trace,
    von_neumann_entropy,
    entanglement_entropy,
    schmidt_decomposition,
    schmidt_rank,
    mutual_information,
    is_pure,
)

plt.rcParams['figure.figsize'] = (8, 4.5)

## 1. Pure vs mixed states

A pure state is a unit vector $|\psi\rangle$ in a Hilbert space. Its density matrix is the rank-1 projector $\rho = |\psi\rangle\langle\psi|$.

A mixed state is a convex combination of pure states, $\rho = \sum_k p_k |\psi_k\rangle\langle\psi_k|$. Operationally, mixed states represent classical ignorance about which pure state you actually have. Crucially, they also arise when you **trace out a subsystem** of a pure joint state — even if the joint state is pure, the marginal state of one subsystem can be mixed, and the mixedness measures the entanglement.

**Test for purity**: $\text{tr}(\rho^2) = 1$ iff $\rho$ is pure.

In [ ]:
# A pure single-qubit state |+> = (|0> + |1>) / sqrt(2)
plus = np.array([1, 1]) / math.sqrt(2)
rho_pure = density_matrix(plus)
print(f'rho_pure = \n{rho_pure.real}')
print(f'tr(rho^2) = {np.trace(rho_pure @ rho_pure).real:.6f}  (pure means 1)')
print(f'is_pure: {is_pure(rho_pure)}')
print()

# A maximally mixed single-qubit state I/2
rho_mixed = np.eye(2) / 2
print(f'rho_mixed = \n{rho_mixed}')
print(f'tr(rho^2) = {np.trace(rho_mixed @ rho_mixed).real:.6f}  (mixed means < 1)')
print(f'is_pure: {is_pure(rho_mixed)}')

## 2. The Bell pair and the partial trace

The Bell state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ is the canonical maximally entangled state of two qubits. **It is itself a pure state** — the joint density matrix has rank 1. But if we only have access to one of the two qubits, the state we see is the partial trace of the joint state over the other qubit:

$$\rho_A = \text{tr}_B\big(|\Phi^+\rangle\langle\Phi^+|\big) = \frac{1}{2}|0\rangle\langle 0| + \frac{1}{2}|1\rangle\langle 1| = \frac{I}{2}.$$

This is the **maximally mixed single-qubit state**. The entire information about the entanglement has been pushed into the correlations between $A$ and $B$, which we can no longer see from $A$ alone.

In [ ]:
bell = np.array([1, 0, 0, 1]) / math.sqrt(2)
rho_AB = density_matrix(bell)
print(f'Joint Bell density matrix (4x4, rank 1):')
print(rho_AB.real)
print(f'is_pure(rho_AB): {is_pure(rho_AB)}')

# Trace out qubit B to get rho_A
rho_A = partial_trace(rho_AB, dims=(2, 2), traced_subsystems=(1,))
print(f'\nReduced density matrix rho_A:')
print(rho_A.real)
print(f'is_pure(rho_A): {is_pure(rho_A)}')
print(f'  -- it is the maximally mixed I/2 state')

## 3. The von Neumann entropy

$$S(\rho) = -\text{tr}(\rho \log \rho) = -\sum_i \lambda_i \log \lambda_i$$

with the convention $0 \log 0 = 0$ and $\lambda_i$ the eigenvalues of $\rho$. Default base is natural log (entropy in *nats*); pass `base='2'` for bits.

**Bounds**:
- $S(\rho) \ge 0$, with equality iff $\rho$ is pure
- $S(\rho) \le \log d$, with equality iff $\rho = I/d$ (the maximally mixed state)

Let's check both bounds against the Bell pair we just constructed:

In [ ]:
S_pure = von_neumann_entropy(rho_AB)
print(f'S(rho_AB)  (pure)        = {S_pure:.10f}    expect 0')

S_mixed = von_neumann_entropy(rho_A)
print(f'S(rho_A)   (max mixed)   = {S_mixed:.10f}    expect log 2 = {math.log(2):.10f}')

S_mixed_bits = von_neumann_entropy(rho_A, base='2')
print(f'S(rho_A)   in bits       = {S_mixed_bits:.10f}    expect 1.0')

# Maximally mixed in higher dimensions
for d in [2, 3, 4, 8]:
    S = von_neumann_entropy(np.eye(d) / d)
    print(f'S(I_{d}/{d})  (max mixed in d={d})  = {S:.6f}    expect log {d} = {math.log(d):.6f}')

**Bell pair carries exactly 1 bit of entanglement.** This is one of the most important canonical results in quantum information.

## 4. The Schmidt decomposition

Every pure bipartite state can be written

$$|\psi\rangle = \sum_{i=1}^{r} \lambda_i\,|u_i\rangle_A \otimes |v_i\rangle_B$$

with orthonormal $\{|u_i\rangle\}$ and $\{|v_i\rangle\}$ and Schmidt coefficients $\lambda_i > 0$ such that $\sum_i \lambda_i^2 = 1$. The number of nonzero $\lambda_i$ is the **Schmidt rank**, which equals the rank of $\rho_A$ (and of $\rho_B$).

**Computationally**, the Schmidt decomposition is just the SVD of the state vector reshaped as a matrix. The Bell pair has Schmidt rank 2:

In [ ]:
coeffs, U, V = schmidt_decomposition(bell, dims=(2, 2))
print(f'Bell pair Schmidt coefficients: {coeffs}')
print(f'Their squares (probabilities): {coeffs**2}')
print(f'Sum of squares: {np.sum(coeffs**2):.10f}  (must be 1)')
print(f'Schmidt rank: {schmidt_rank(bell, dims=(2, 2))}')
print()

# Compare with a product state
product = np.kron([1, 0], [0, 1])
print(f'Product state |0> tensor |1>:')
print(f'  Schmidt rank: {schmidt_rank(product, dims=(2, 2))}')
print(f'  Entanglement entropy: {entanglement_entropy(product, dims=(2, 2)):.10f}  (expect 0)')

## 5. Entanglement vs entanglement: a smooth interpolation

Let's interpolate between a product state and a Bell state and watch the entanglement entropy rise smoothly. Define

$$|\psi(\theta)\rangle = \cos\theta\,|00\rangle + \sin\theta\,|11\rangle.$$

For $\theta = 0$ this is the product state $|00\rangle$. For $\theta = \pi/4$ it is the Bell pair. For $\theta = \pi/2$ it is the product state $|11\rangle$.

The entanglement entropy is

$$S(\theta) = -\cos^2\theta \log\cos^2\theta - \sin^2\theta \log\sin^2\theta,$$

which is the Shannon entropy of the distribution $(\cos^2\theta, \sin^2\theta)$. It hits its maximum $\log 2$ exactly at $\theta = \pi/4$.

In [ ]:
thetas = np.linspace(0, math.pi / 2, 200)
Ss = np.array(
    [
        entanglement_entropy(
            np.array([math.cos(t), 0.0, 0.0, math.sin(t)]), dims=(2, 2)
        )
        for t in thetas
    ]
)

fig, ax = plt.subplots()
ax.plot(thetas, Ss, color='#0050a0', lw=1.6)
ax.axhline(math.log(2), color='#a40000', lw=1.0, ls='--', label='$\\log 2$ (Bell maximum)')
ax.axvline(math.pi / 4, color='#a40000', lw=1.0, ls=':', label='$\\theta = \\pi/4$ (Bell pair)')
ax.set_xlabel(r'$\theta$')
ax.set_ylabel(r'$S(\theta)$ (nats)')
ax.set_title(r'Entanglement entropy of $\cos\theta\,|00\rangle + \sin\theta\,|11\rangle$')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. The 3-qubit GHZ state and its bipartitions

The Greenberger-Horne-Zeilinger state for 3 qubits is

$$|\text{GHZ}\rangle = \frac{1}{\sqrt 2}\big(|000\rangle + |111\rangle\big).$$

It is genuinely tripartite-entangled, but its **bipartite** structure is simple: tracing out *any* one qubit leaves the same separable mixed state $\frac{1}{2}(|00\rangle\langle 00| + |11\rangle\langle 11|)$ on the other two. So every 1-vs-2 bipartition has entanglement entropy $\log 2$:

In [ ]:
ghz = np.zeros(8)
ghz[0] = 1.0  # |000>
ghz[7] = 1.0  # |111>
ghz = ghz / math.sqrt(2)
rho_ghz = density_matrix(ghz)

for traced_qubit in [0, 1, 2]:
    other = tuple(q for q in [0, 1, 2] if q != traced_qubit)
    rho_red = partial_trace(rho_ghz, dims=(2, 2, 2), traced_subsystems=other)
    S = von_neumann_entropy(rho_red)
    print(f'Trace out qubits {other}, keep qubit {traced_qubit}: S = {S:.10f}    expect log 2')

# Also: trace out one qubit and keep the pair
for traced_qubit in [0, 1, 2]:
    rho_pair = partial_trace(rho_ghz, dims=(2, 2, 2), traced_subsystems=(traced_qubit,))
    S = von_neumann_entropy(rho_pair)
    print(f'Trace out qubit {traced_qubit}, keep pair: S = {S:.10f}    expect log 2')

All six bipartitions give exactly $\log 2$. The symmetry under $\mathbb{Z}_3$ permutation of the three qubits is fully manifest.

## 7. Quantum mutual information

The quantum mutual information between two subsystems is

$$I(A : B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB}).$$

It is non-negative for any joint state and zero iff the joint state factorises ($\rho_{AB} = \rho_A \otimes \rho_B$). For a *pure* entangled state $|\psi_{AB}\rangle$, the joint entropy $S(\rho_{AB}) = 0$, so $I(A:B) = 2 S(\rho_A) = 2$ (entanglement entropy).

In [ ]:
# Bell pair: I(A:B) = 2 log 2
I_bell = mutual_information(rho_AB, dims=(2, 2), subsystem_A=(0,), subsystem_B=(1,))
print(f'Bell pair I(A:B) = {I_bell:.10f}    expect 2 log 2 = {2 * math.log(2):.10f}')

# Product state: I(A:B) = 0
rho_prod = density_matrix(np.kron([1, 0], [0, 1]))
I_prod = mutual_information(rho_prod, dims=(2, 2), subsystem_A=(0,), subsystem_B=(1,))
print(f'Product state I(A:B) = {I_prod:.10f}    expect 0')

# GHZ: I(qubit_0 : qubit_1) - they share one bit of correlation through the GHZ structure
I_ghz = mutual_information(rho_ghz, dims=(2, 2, 2), subsystem_A=(0,), subsystem_B=(1,))
print(f'GHZ I(0:1) = {I_ghz:.10f}    expect log 2 (classical correlation only)')

Notice the GHZ result is $\log 2$, not $2 \log 2$. The GHZ state has the same kind of bipartite mutual information as a pair of *classically correlated* coins (each in the state $\frac12 |00\rangle\langle 00| + \frac12 |11\rangle\langle 11|$): one bit of correlation. The 'extra' quantum content of GHZ shows up in its tripartite structure, not in any bipartite reduction.

## 8. The bridge to holography

Recall from Phases 1, 3, 4 that the Bekenstein-Hawking entropy of a black hole is

$$S_{BH} = \frac{A}{4 G_N} = \frac{\text{horizon area}}{4 \hbar G_N}\;\text{(in natural units)}.$$

We computed this directly from the metric for both Schwarzschild ($S = 4\pi M^2$) and Kerr ($S = \pi(r_+^2 + a^2)$).

**The deep claim of holographic entanglement entropy** (Ryu-Takayanagi 2006, Hubeny-Rangamani-Takayanagi 2007, Penington 2019, Almheiri et al 2019) is that this is *literally* an entanglement entropy: the Bekenstein-Hawking formula is a special case of

$$S_A^{\text{boundary CFT}} = \frac{\text{Area}(\gamma_A^{\text{bulk}})}{4 G_N},$$

where $\gamma_A$ is a *minimal surface* in the bulk anchored to the boundary of region $A$. The horizon is one such minimal surface — the one corresponding to the entire boundary as the region $A$.

This is the holographic principle in its sharpest form. To make any of it quantitative we need:
- A way to compute $S_A$ on the QFT side (**Phase 6 — this notebook**)
- A way to find minimal surfaces in AdS bulk geometries (Phase 7-8)
- A way to handle the quantum corrections that become important when entanglement wedges change topology (Phase 9, the *island formula*)

The von Neumann entropy machinery we built in this notebook is **the fundamental primitive** for everything that follows. Without it, we cannot test any holographic claim.

## 9. Phase 6 gate checks

Before moving on to Phase 7 the following must hold.

In [ ]:
# (1) Bell pair entanglement entropy = log 2 (in nats), 1 bit
S = entanglement_entropy(np.array([1, 0, 0, 1]) / math.sqrt(2), dims=(2, 2))
assert math.isclose(S, math.log(2), abs_tol=1e-12)
S_bits = entanglement_entropy(np.array([1, 0, 0, 1]) / math.sqrt(2), dims=(2, 2), base='2')
assert math.isclose(S_bits, 1.0, abs_tol=1e-12)

# (2) Product state has zero entanglement
S_prod = entanglement_entropy(np.kron([1, 0], [0, 1]), dims=(2, 2))
assert math.isclose(S_prod, 0.0, abs_tol=1e-12)

# (3) Maximally mixed state I/d has entropy log d
for d in [2, 3, 4, 8]:
    assert math.isclose(von_neumann_entropy(np.eye(d) / d), math.log(d), abs_tol=1e-12)

# (4) GHZ has every 1-vs-2 bipartition equal to log 2
for traced in [(0,), (1,), (2,)]:
    rho_red = partial_trace(rho_ghz, dims=(2, 2, 2), traced_subsystems=traced)
    assert math.isclose(von_neumann_entropy(rho_red), math.log(2), abs_tol=1e-12)

# (5) Schmidt rank distinguishes product from entangled
assert schmidt_rank(np.array([1, 0, 0, 1]) / math.sqrt(2), dims=(2, 2)) == 2
assert schmidt_rank(np.kron([1, 0], [0, 1]), dims=(2, 2)) == 1

# (6) Bell mutual information = 2 log 2
I = mutual_information(rho_AB, dims=(2, 2), subsystem_A=(0,), subsystem_B=(1,))
assert math.isclose(I, 2 * math.log(2), abs_tol=1e-12)

# (7) Pure state via density matrix has trace(rho^2) = 1
rho = density_matrix(np.array([0.6, 0.8]))
assert math.isclose(np.trace(rho @ rho).real, 1.0, abs_tol=1e-12)

# (8) Schmidt coefficients squared sum to 1
rng = np.random.default_rng(42)
psi = rng.standard_normal(12) + 1j * rng.standard_normal(12)
psi = psi / np.linalg.norm(psi)
coeffs, _, _ = schmidt_decomposition(psi, dims=(3, 4))
assert math.isclose(np.sum(coeffs ** 2), 1.0, abs_tol=1e-12)

# (9) Additivity for product density matrices
rho_A = np.diag([0.6, 0.4])
rho_B = np.diag([0.5, 0.3, 0.2])
S_A = von_neumann_entropy(rho_A)
S_B = von_neumann_entropy(rho_B)
S_AB = von_neumann_entropy(np.kron(rho_A, rho_B))
assert math.isclose(S_A + S_B, S_AB, abs_tol=1e-12)

print('ALL PHASE 6 GATE CHECKS PASSED')

---

## What's next — Phase 7 teaser

Phase 7 enters **AdS/CFT correspondence** at the level needed to set up the Ryu-Takayanagi computation in Phase 8. Concrete deliverables:

1. The pure AdS$_{d+1}$ metric in global coordinates and Poincaré coordinates
2. The conformal boundary at $z = 0$ in Poincaré coordinates
3. The geodesic length between two boundary points (the simplest holographic 'minimal surface' is a geodesic in AdS$_3$ between two points on the boundary)
4. Verification: in AdS$_3$/CFT$_2$, the geodesic length divided by $4 G_N$ exactly reproduces the entanglement entropy of an interval in a 2D CFT, $S = (c/3) \log(L/\epsilon)$

The connection to Phase 6 is that **the right-hand side of the RT formula will be computed using `von_neumann_entropy` from this notebook**, applied to the reduced density matrix of a 2D CFT interval. The RT formula then says it equals the geodesic length in pure AdS$_3$. We will verify this numerically end-to-end.